# Load, apply, and compare saved constituent models

This notebook reconstructs each architecture from its JSON configuration and weights-only
checkpoint. Direct comparisons are allowed only on the same dataset and split fingerprint.


## 1. Environment


In [ ]:
import importlib.util
required = ['numpy', 'pandas', 'pyarrow', 'matplotlib', 'sklearn', 'torch', 'tqdm']
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(
        f"Missing packages: {missing}. From a terminal in this directory run "
        "./setup_student_env.sh (or use --current inside an existing henv), "
        "restart Jupyter from that henv, and select its registered kernel."
    )

import json, os, time
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm.auto import tqdm
import qg_constituent_ml as qg

DEVICE = qg.choose_device()
RUN_MODE = os.getenv('QG_RUN_MODE', 'quick')
SOURCE = Path(os.getenv('QG_INPUT_PATH', 'data/inclusive_jets.parquet'))
print(f'PyTorch {torch.__version__}; built with CUDA {torch.version.cuda}')
print(f'device={DEVICE}' + (f'; GPU={torch.cuda.get_device_name(0)}' if DEVICE.type == 'cuda' else ''))
print(f'run mode={RUN_MODE}; source={SOURCE}')


In [ ]:
prepared=qg.prepare_dataset(SOURCE); manifest=qg.load_manifest(prepared)
bundles=qg.discover_bundles(dataset_fingerprint=manifest['source_sha256'])
if not bundles: raise FileNotFoundError('Train at least one architecture notebook first.')
print('\n'.join(map(str,bundles)))


## 2. Reload checkpoints and reproduce their predictions


In [ ]:
from sklearn.metrics import roc_curve
rows=[]; curves={}
for bundle in tqdm(bundles, desc='Evaluating saved models', unit='model'):
    model,config=qg.load_model_bundle(bundle,DEVICE)
    loaders=qg.make_loaders(prepared,config['architecture'],config['mode'])
    pred=qg.predict(model,loaders[2],DEVICE,progress=True,
                    description=f"Evaluating {config['architecture']}"); saved=np.load(bundle/'predictions.npz')
    assert np.array_equal(pred['event_ids'],saved['event_ids']) and np.array_equal(pred['jet_ids'],saved['jet_ids'])
    assert np.allclose(pred['scores'],saved['scores'],atol=2e-5)
    metrics=qg.binary_metrics(pred['labels'],pred['scores']); metrics['model']=f"{config['architecture']} ({config['mode']})"
    metrics['parameters']=sum(p.numel() for p in model.parameters()); metrics['bundle']=str(bundle); rows.append(metrics)
    curves[metrics['model']]=(*roc_curve(pred['labels'],pred['scores'])[:2],metrics['roc_auc'])
import pandas as pd
results=pd.DataFrame(rows).sort_values('roc_auc',ascending=False); display(results)


In [ ]:
fig,ax=plt.subplots(figsize=(7,5))
for name,(fpr,tpr,auc) in curves.items(): ax.plot(tpr,1/np.clip(fpr,1e-3,None),label=f'{name}: {auc:.3f}')
ax.set(xlabel='quark efficiency',ylabel='gluon rejection',yscale='log',title='Common held-out comparison')
ax.legend(); plt.tight_layout(); plt.show()


## 3. Apply a model to another sample

Set `QG_EVAL_PATH` before launching to evaluate a different compatible Parquet file. The
saved training normalization is reused; it must never be refitted on the evaluation sample.


In [ ]:
EVAL_PATH=os.getenv('QG_EVAL_PATH')
if EVAL_PATH:
    evaluation,config=qg.predict_parquet(bundles[0],EVAL_PATH,device=DEVICE)
    labeled=evaluation['labels'] >= 0
    print(f"Scored all {len(evaluation['scores']):,} jets from {EVAL_PATH}; {labeled.sum():,} have quark/gluon labels")
    if labeled.any(): print(json.dumps(qg.binary_metrics(evaluation['labels'][labeled],evaluation['scores'][labeled]),indent=2))
else: print('QG_EVAL_PATH is unset; common-test comparison complete.')
